# Investor Classifier, Part II - Model Training & Selection

This notebook trains and cross-validates classification algorithms to predict investor commitment decisions. Uses the pre-engineered dataset from Part I.

## 1. Redundant Features & Dummy Variables

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

investor_data = pd.read_csv('investor_data_2.csv')
investor_data.head(3)

In [ ]:
investor_data = investor_data.drop(['invite_tier', 'fee_share', 'invite'], axis=1)
investor_data.shape

In [ ]:
investor_data = pd.get_dummies(investor_data)
investor_data.shape

In [ ]:
investor_data.head(1)

In [ ]:
investor_data = investor_data.drop('commit_Commit', axis=1)

target = investor_data.commit_Decline
inputs = investor_data.drop('commit_Decline', axis=1)

## 2. Stratified Random Sampling

In [ ]:
from sklearn.model_selection import train_test_split

split_list = train_test_split(inputs, target, test_size=0.2, random_state=1, stratify=investor_data.commit_Decline)
input_train, input_test, target_train, target_test = split_list

print(f"Training inputs: {input_train.shape}")
print(f"Testing inputs: {input_test.shape}")
print(f"Training target: {target_train.shape}")
print(f"Testing target: {target_test.shape}")

## 3. Pipelines & Hyperparameter Grids

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

pipelines = {
    'l1': make_pipeline(StandardScaler(), LogisticRegression(penalty='l1', random_state=1, solver='liblinear')),
    'l2': make_pipeline(StandardScaler(), LogisticRegression(penalty='l2', random_state=1, solver='liblinear')),
    'rf': make_pipeline(StandardScaler(), RandomForestClassifier(random_state=1)),
    'gb': make_pipeline(StandardScaler(), GradientBoostingClassifier(random_state=1))
}

l1_hyperparameters = {'logisticregression__C': [0.1, 1, 10]}
l2_hyperparameters = {'logisticregression__C': [0.1, 1, 10]}
rf_hyperparameters = {
    'randomforestclassifier__n_estimators': [100, 200],
    'randomforestclassifier__max_features': ['auto', 0.3, 0.6]
}
gb_hyperparameters = {
    'gradientboostingclassifier__n_estimators': [100, 200],
    'gradientboostingclassifier__learning_rate': [0.05, 0.1, 0.2],
    'gradientboostingclassifier__max_depth': [1, 3, 5]
}

hyperparameters = {
    'l1': l1_hyperparameters,
    'l2': l2_hyperparameters,
    'rf': rf_hyperparameters,
    'gb': gb_hyperparameters
}

for key, value in pipelines.items():
    print(key, type(value))

## 4. Cross-Validation

In [ ]:
models = {}

for key in pipelines.keys():
    models[key] = GridSearchCV(pipelines[key], hyperparameters[key], cv=5)

for key in models.keys():
    models[key].fit(input_train, target_train)
    print(key, 'is trained and tuned.')

## 5. Model Selection

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, auc

for key in models.keys():
    pred = models[key].predict(input_test)
    fpr, tpr, thresholds = roc_curve(target_test, pred)
    print(key)
    print('AUROC =', round(auc(fpr, tpr), 4))
    print('---')

In [ ]:
# Confusion matrix for the winning model (Gradient Boosting)
pred = models['gb'].predict(input_test)
cm = confusion_matrix(target_test, pred)
print("Confusion Matrix:")
print(cm)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Commit', 'Decline'],
            yticklabels=['Commit', 'Decline'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Gradient Boosting Classifier - Confusion Matrix')
plt.show()